<a href="https://colab.research.google.com/github/ThibaultHareau/Projects/blob/project%2Fweatherforecast/MachineLearning/WeatherForecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Weather Forecast Project**

This notebook attempts to predict the weather based on the data of the previous days.

The dataset is available here: https://www.google.com/url?q=http%3A%2F%2Fclimate.weather.gc.ca%2Fhistorical_data%2Fsearch_historic_data_e.html

# 0. Import the prerequisites

In [70]:
import pandas as pd
import keras

# data visualization
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import seaborn as sns

# 1. Download the data

In [49]:
url = "https://gist.githubusercontent.com/adrn/6455c48cb556d2f6f939c1e55b2308f8/raw/dedf6fd1989ab46baeab51058c788cca8aba4ca3/weather_cache_sm.csv"
nyc_weather_data = pd.read_csv(url)

# 2. Transform the data

In [52]:
nyc_weather_data_filtered = nyc_weather_data[["day", "month", "year", "temperatureHigh"]]
nyc_weather_data_filtered["rank"] = (nyc_weather_data_filtered.sort_values(by=["year", "month", "day"]))[["year","month", "day"]].apply(tuple,axis=1).rank(method='first',ascending=True).astype(int)
df = nyc_weather_data_filtered[["rank", "temperatureHigh"]]
display(df)

<ipython-input-52-1e8a8d67d7d2>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nyc_weather_data_filtered["rank"] = (nyc_weather_data_filtered.sort_values(by=["year", "month", "day"]))[["year","month", "day"]].apply(tuple,axis=1).rank(method='first',ascending=True).astype(int)


,rank,temperatureHigh
0,1,32.93
1,2,27.69
2,3,21.68
3,4,34.15
4,5,33.98
...,...,...
19255,19256,69.28
19256,19257,69.75
19257,19258,78.82
19258,19259,82.91


In [87]:
lookback_days = 5
to_merge = df[["rank", "temperatureHigh"]]
previous_days_temp = df
for i in range(1, lookback_days+1):
  previous_days_temp[f"rank-{i}"] = previous_days_temp["rank"]-i
  previous_days_temp = pd.merge(previous_days_temp, to_merge.rename(columns={"rank": f"rank-{i}", "temperatureHigh": f"temperatureHigh_-{i}"}), how="left", on=[f"rank-{i}"])
  previous_days_temp = previous_days_temp.drop(f"rank-{i}", axis=1)

previous_days_temp = previous_days_temp.loc[df['rank'] > lookback_days]
previous_days_temp = previous_days_temp.drop("rank", axis=1)
display(previous_days_temp)

<ipython-input-87-a275a5424ed2>:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,temperatureHigh,temperatureHigh_-1,temperatureHigh_-2,temperatureHigh_-3,temperatureHigh_-4,temperatureHigh_-5
5,21.00,33.98,34.15,21.68,27.69,32.93
6,28.39,21.00,33.98,34.15,21.68,27.69
7,26.29,28.39,21.00,33.98,34.15,21.68
8,12.66,26.29,28.39,21.00,33.98,34.15
9,13.50,12.66,26.29,28.39,21.00,33.98
...,...,...,...,...,...,...
19255,69.28,76.94,75.89,82.24,73.45,69.84
19256,69.75,69.28,76.94,75.89,82.24,73.45
19257,78.82,69.75,69.28,76.94,75.89,82.24
19258,82.91,78.82,69.75,69.28,76.94,75.89


# 3. Data Exploration

In [88]:
#@title View correlation matrix
previous_days_temp.corr(numeric_only = True)

,temperatureHigh,temperatureHigh_-1,temperatureHigh_-2,temperatureHigh_-3,temperatureHigh_-4,temperatureHigh_-5
temperatureHigh,1.000000,0.913719,0.857441,0.834532,0.821607,0.815351
temperatureHigh_-1,0.913719,1.000000,0.913724,0.857452,0.834572,0.821649
temperatureHigh_-2,0.857441,0.913724,1.000000,0.913724,0.857477,0.834599
temperatureHigh_-3,0.834532,0.857452,0.913724,1.000000,0.913743,0.857496
temperatureHigh_-4,0.821607,0.834572,0.857477,0.913743,1.000000,0.913752
temperatureHigh_-5,0.815351,0.821649,0.834599,0.857496,0.913752,1.000000


# 4. Train Model

In [89]:
#@title Define plotting functions

def make_plots(df, feature_names, label_name, model_output, sample_size=200):

  random_sample = df.sample(n=sample_size).copy()
  random_sample.reset_index()
  weights, bias, epochs, rmse = model_output

  is_2d_plot = len(feature_names) == 1
  model_plot_type = "scatter" if is_2d_plot else "surface"
  fig = make_subplots(rows=1, cols=2,
                      subplot_titles=("Loss Curve", "Model Plot"),
                      specs=[[{"type": "scatter"}, {"type": model_plot_type}]])

  plot_data(random_sample, feature_names, label_name, fig)
  plot_model(random_sample, feature_names, weights, bias, fig)
  plot_loss_curve(epochs, rmse, fig)

  fig.show()
  return

def plot_loss_curve(epochs, rmse, fig):
  curve = px.line(x=epochs, y=rmse)
  curve.update_traces(line_color='#ff0000', line_width=3)

  fig.append_trace(curve.data[0], row=1, col=1)
  fig.update_xaxes(title_text="Epoch", row=1, col=1)
  fig.update_yaxes(title_text="Root Mean Squared Error", row=1, col=1, range=[rmse.min()*0.8, rmse.max()])

  return

def plot_data(df, features, label, fig):
  if len(features) == 1:
    scatter = px.scatter(df, x=features[0], y=label)
  else:
    scatter = px.scatter_3d(df, x=features[0], y=features[1], z=label)

  fig.append_trace(scatter.data[0], row=1, col=2)
  if len(features) == 1:
    fig.update_xaxes(title_text=features[0], row=1, col=2)
    fig.update_yaxes(title_text=label, row=1, col=2)
  else:
    fig.update_layout(scene1=dict(xaxis_title=features[0], yaxis_title=features[1], zaxis_title=label))

  return

def plot_model(df, features, weights, bias, fig):
  df['FARE_PREDICTED'] = bias[0]

  for index, feature in enumerate(features):
    df['FARE_PREDICTED'] = df['FARE_PREDICTED'] + weights[index][0] * df[feature]

  if len(features) == 1:
    model = px.line(df, x=features[0], y='FARE_PREDICTED')
    model.update_traces(line_color='#ff0000', line_width=3)
  else:
    z_name, y_name = "FARE_PREDICTED", features[1]
    z = [df[z_name].min(), (df[z_name].max() - df[z_name].min()) / 2, df[z_name].max()]
    y = [df[y_name].min(), (df[y_name].max() - df[y_name].min()) / 2, df[y_name].max()]
    x = []
    for i in range(len(y)):
      x.append((z[i] - weights[1][0] * y[i] - bias[0]) / weights[0][0])

    plane=pd.DataFrame({'x':x, 'y':y, 'z':[z] * 3})

    light_yellow = [[0, '#89CFF0'], [1, '#FFDB58']]
    model = go.Figure(data=go.Surface(x=plane['x'], y=plane['y'], z=plane['z'],
                                      colorscale=light_yellow))

  fig.add_trace(model.data[0], row=1, col=2)

  return

def model_info(feature_names, label_name, model_output):
  weights = model_output[0]
  bias = model_output[1]

  nl = "\n"
  header = "-" * 80
  banner = header + nl + "|" + "MODEL INFO".center(78) + "|" + nl + header

  info = ""
  equation = label_name + " = "

  for index, feature in enumerate(feature_names):
    info = info + "Weight for feature[{}]: {:.3f}\n".format(feature, weights[index][0])
    equation = equation + "{:.3f} * {} + ".format(weights[index][0], feature)

  info = info + "Bias: {:.3f}\n".format(bias[0])
  equation = equation + "{:.3f}\n".format(bias[0])

  return banner + nl + info + nl + equation

print("SUCCESS: defining plotting functions complete.")

SUCCESS: defining plotting functions complete.


In [95]:
#@title Code - Define ML functions

def build_model(my_learning_rate, num_features):
  """Create and compile a simple linear regression model."""
  # Describe the topography of the model.
  # The topography of a simple linear regression model
  # is a single node in a single layer.
  inputs = keras.Input(shape=(num_features,))
  outputs = keras.layers.Dense(units=1)(inputs)
  model = keras.Model(inputs=inputs, outputs=outputs)

  # Errors ["mean_squared_error", "mean_absolute_error"]

  # Compile the model topography into code that Keras can efficiently
  # execute. Configure training to minimize the model's mean squared error.
  model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=my_learning_rate),
                loss="mean_absolute_error",
                metrics=[keras.metrics.RootMeanSquaredError()])

  return model


def train_model(model, df, features, label, epochs, batch_size):
  """Train the model by feeding it data."""

  # Feed the model the feature and the label.
  # The model will train for the specified number of epochs.
  # input_x = df.iloc[:,1:3].values
  # df[feature]
  history = model.fit(x=features,
                      y=label,
                      batch_size=batch_size,
                      epochs=epochs)

  # Gather the trained model's weight and bias.
  trained_weight = model.get_weights()[0]
  trained_bias = model.get_weights()[1]

  # The list of epochs is stored separately from the rest of history.
  epochs = history.epoch

  # Isolate the error for each epoch.
  hist = pd.DataFrame(history.history)

  # To track the progression of training, we're going to take a snapshot
  # of the model's root mean squared error at each epoch.
  rmse = hist["root_mean_squared_error"]

  return trained_weight, trained_bias, epochs, rmse


def run_experiment(df, feature_names, label_name, learning_rate, epochs, batch_size):

  print('INFO: starting training experiment with features={} and label={}\n'.format(feature_names, label_name))

  num_features = len(feature_names)

  features = df.loc[:, feature_names].values
  label = df[label_name].values

  model = build_model(learning_rate, num_features)
  model_output = train_model(model, df, features, label, epochs, batch_size)

  print('\nSUCCESS: training experiment complete\n')
  print('{}'.format(model_info(feature_names, label_name, model_output)))
  make_plots(df, feature_names, label_name, model_output)

  return model

print("SUCCESS: defining linear regression functions complete.")

SUCCESS: defining linear regression functions complete.


In [96]:
#@title Experiment

# The following variables are the hyperparameters.
learning_rate = 0.001
epochs = 100
batch_size = 100

features = [f"temperatureHigh_-{i}" for i in range(1, lookback_days+1)]
label = 'temperatureHigh'

model_2 = run_experiment(previous_days_temp, features, label, learning_rate, epochs, batch_size)

INFO: starting training experiment with features=['temperatureHigh_-1', 'temperatureHigh_-2', 'temperatureHigh_-3', 'temperatureHigh_-4', 'temperatureHigh_-5'] and label=temperatureHigh

Epoch 1/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 37.8220 - root_mean_squared_error: 43.0643
Epoch 2/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 13.3686 - root_mean_squared_error: 17.0428
Epoch 3/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12.5007 - root_mean_squared_error: 15.9472
Epoch 4/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 11.6026 - root_mean_squared_error: 14.7471
Epoch 5/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 10.8730 - root_mean_squared_error: 13.8363
Epoch 6/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 9.9540 - root_mean_squared_error: 12.7011
Epoch 7/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9.2789 - root_mean_squared_error: 11.8128
Epoch 8/100
193/193 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8.6925 - root_mean_squared